

# Safeguarding your Strands Agents with Amazon Bedrock Guardrails

## Overview
In this example we will guide you through how to create your first Strands Agent using [Amazon Bedrock Guardrails](https://aws.amazon.com/bedrock/guardrails/) to safeguard the underlying Amazon Bedrock LLM model.

Amazon Bedrock Guardrails provides configurable safeguards to help safely build generative AI applications at scale. With a consistent and standard approach used across a wide range of foundation models (FMs) including FMs supported in Amazon Bedrock, fine-tuned models, and models hosted outside of Amazon Bedrock, Guardrails delivers industry-leading safety protections. 

With Strands Agents, you can add Amazon Bedrock Guardrails directly to your Amazon Bedrock models. If you are not using Amazon Bedrock Models, you can use the [Apply Guardrail API](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-use-independent-api.html) to safeguard any model. In this case, you need to build the pipelines using the advance processing capabilities you just learned.


## Agent Details
<div style="float: left; margin-right: 20px;">
    
|Feature             |Description                                        |
|--------------------|---------------------------------------------------|
|AWS Services used   |Amazon Bedrock Guardrails                          |
|Custom tools created|get_customer_profile, list_customer_purchases, list_customer_tickets, update_customer_profile|
|Agent Structure     |Single agent architecture                          |

</div>


## Architecture

<div style="text-align:left">
    <img src="images/architecture.png" width="85%" />
</div>

## Key Features
* **Single agent architecture**: this example creates a single agent that interacts with built-in and custom tools
* **Custom tools**: learn how to create your own tools
* **Amazon Bedrock Model as underlying LLM**: Used Anthropic Claude Sonnet 4.5 from Amazon Bedrock as the underlying LLM model
* **Amazon Bedrock Guardrails as underlying safeguard**: Use Amazon Bedrock Guardrails to safe guard your agent application

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Guardrails

Let's now install the requirement packages for our Strands Agent

In [ ]:
!pip install -U -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages

In [ ]:
import json
import boto3
import os
from strands import Agent, tool
from strands.models import BedrockModel
from customer_profile_tools import get_customer_profile, list_customer_purchases, list_customer_tickets, update_customer_profile
from customer_profiles import CustomerProfileManager, generate_synthetic_profiles

### Generating data

Let's generate some synthetic data for this example. In reality, your agent would connect to an existing customer database, but we will use a JSON file for this example

In [ ]:
 # Generate synthetic profiles if needed
profile_manager = CustomerProfileManager()
if not os.path.exists("customer_profiles.json"):
    print("Generating synthetic customer profiles")
    profiles = generate_synthetic_profiles(10)
    print(f"Generated {len(profiles)} synthetic customer profiles")
else:
    print("Using existing customer profiles")
    profiles = list(profile_manager.profiles.values())
    print(f"Loaded {len(profiles)} customer profiles")

# Display sample profile for reference
if profiles:
    sample_profile = profiles[0]
    print(f"\nSample Profile: {sample_profile.name}")
    print(f"Email: {sample_profile.email}")
    print(f"Customer ID: {sample_profile.customer_id}")
    print(f"Location: {sample_profile.state}, {sample_profile.country}")
    print(f"Purchases: {len(sample_profile.purchase_history)}")
    print(f"Support Tickets: {len(sample_profile.support_tickets)}")

## Creating an Amazon Bedrock Guardrail

Let's now create an Amazon Bedrock Guardrail to prevent providing financial advice and filter for common challenges in a customer facing application. 

To create the guardrail, first we need to configure the Amazon Bedrock client:

In [ ]:
# Configure AWS clients
bedrock_client = boto3.client('bedrock')
bedrock_runtime = boto3.client('bedrock-runtime')

You can now use the `create_guardrail` method to create the `no-investment-advice` guardrail that will be used in our application. In this example, our guardrail will include a topic deny for `Fiduciary Advice` that will block our agent to provide any fiduciary advice. The guardrail will also contain some basic content policy to filter inappropriate content and a word policy configuration to detect specific pre-defined words.

In [ ]:
# Create a Bedrock guardrail
response = bedrock_client.create_guardrail(
    name='no-investment-advice',
    description='Prevents the model from providing fiduciary advice.',
    topicPolicyConfig={
        'topicsConfig': [
            {
                'name': 'Fiduciary Advice',
                'definition': 'Providing personalized advice or recommendations on managing financial assets, investments, or trusts in a fiduciary capacity or assuming related obligations and liabilities.',
                'examples': [
                    'What stocks should I invest in for my retirement?',
                    'Is it a good idea to put my money in a mutual fund?',
                    'How should I allocate my 401(k) investments?',
                    'What type of trust fund should I set up for my children?',
                    'Should I hire a financial advisor to manage my investments?'
                ],
                'type': 'DENY'
            }
        ]
    },
    contentPolicyConfig={
        'filtersConfig': [
            {
                'type': 'SEXUAL',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'VIOLENCE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'HATE',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'INSULTS',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'MISCONDUCT',
                'inputStrength': 'HIGH',
                'outputStrength': 'HIGH'
            },
            {
                'type': 'PROMPT_ATTACK',
                'inputStrength': 'HIGH',
                'outputStrength': 'NONE'
            }
        ]
    },
    wordPolicyConfig={
        'wordsConfig': [
            {'text': 'fiduciary advice'},
            {'text': 'investment recommendations'},
            {'text': 'stock picks'},
            {'text': 'financial planning guidance'},
            {'text': 'portfolio allocation advice'},
            {'text': 'retirement fund suggestions'},
            {'text': 'wealth management tips'},
            {'text': 'trust fund setup'},
            {'text': 'investment strategy'},
            {'text': 'financial advisor recommendations'}
        ],
        'managedWordListsConfig': [
            {
                'type': 'PROFANITY'
            }
        ]
    },
    blockedInputMessaging='I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including any financial, or restricted details.',
    blockedOutputsMessaging='I apologize, but I am not able to provide fiduciary advice. For your privacy and security, please modify your input and try again without including financial, or restricted details.',
)

# Print the response to get the guardrail ID
print("Guardrail ID:", response.get('guardrailId'))
print("Guardrail ARN:", response.get('guardrailArn'))

# Store the guardrail ID for later use


In [ ]:
guardrail_id = response.get('guardrailId')
guardrail_version = "DRAFT"  # Initial version is always 1

### Testing the guardrail directly

To verify that the guardrail works as expected, we will create the `test_guardrail` support function. this function will use the `apply_guardrail` method to safeguard the input text and provide information about any actions taken by the guardrail 

In [ ]:
# Test function to check if input/output is blocked by guardrail
def test_guardrail(text, source_type='INPUT'):
      response = bedrock_runtime.apply_guardrail(
          guardrailIdentifier=guardrail_id,
          guardrailVersion=guardrail_version,
          source=source_type,  # can be 'INPUT' or 'OUTPUT'
          content=[{"text": {"text": text}}]
      )

      # New response format uses different fields
      print(f"Action: {response.get('action')}")
      print(f"Action Reason: {response.get('actionReason', 'None')}")

      # Check if content was blocked
      is_blocked = response.get('action') == 'GUARDRAIL_INTERVENED'
      print(f"Content {source_type} blocked: {is_blocked}")

      if is_blocked:
          # Print topic policies that were triggered
          assessments = response.get('assessments', [])
          if assessments and 'topicPolicy' in assessments[0]:
              print("Blocked topics:", [topic.get('name') for topic in
  assessments[0]['topicPolicy'].get('topics', [])
                                       if topic.get('action') == 'BLOCKED'])

          # Print the modified output if available
          if 'outputs' in response and response['outputs']:
              print("Modified content:", response['outputs'][0].get('text', 'None'))

      return response

# Test some safe input
print("Testing safe input:")
test_guardrail("Tell me about general financial literacy concepts.")

# Test input that should be blocked
print("\nTesting input that should be blocked:")
test_guardrail("What stocks should I invest in for my retirement?")

## Integrating with Strands Agent

Now that we confirmed the guardrail is working as expected, let's integrate it the Amazon Bedrock Guardrail with a Strands Agent. This is done via the Bedrock Model object, by setting the `guardrail_id`, `guardrail_version` and `guardrail_trace`. Once the model object is created you can use it to create your agent. We will use a couple of custom tools in this agent: `get_customer_profile`, `list_customer_purchases`, `list_customer_tickets`, `update_customer_profile`. To see their implementation check the `customer_profile_tools.py` file

In [ ]:
guardrail_id, guardrail_version

In [ ]:
# Create a Bedrock model with guardrail configuration
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    guardrail_id=guardrail_id,
    guardrail_version=guardrail_version,
    # Enable trace info for debugging
    guardrail_trace="enabled"
)

# Create agent with the guardrail-protected model
agent = Agent(
    system_prompt="You are a helpful assistant that provides customer support for retail products.",
    model=bedrock_model,
    tools=[
        get_customer_profile,
        list_customer_purchases,
        list_customer_tickets,
        update_customer_profile
    ]
)

### Testing the Strands Agent with Guardrails

Let's test our agent with both safe and risky inputs. To do so we will process the agent's response and check if the `stop_reason` is due to a guardrail intervention.

In [ ]:
# Helper function to test the agent and check for guardrail interventions
def test_agent_with_guardrail(prompt):
    print(f"\nUser: {prompt}")

    # Get agent response
    response = agent(prompt)

    # Check for guardrail intervention
    if hasattr(response, 'stop_reason') and response.stop_reason == "guardrail_intervened":
        print("\n ⚠️ GUARDRAIL INTERVENED!")
        #print(f"Response: {response}")
    else:
        return response

In [ ]:
# Test with a safe question 
test_agent_with_guardrail(
    " what is my latest purchase my Customer ID is CUST100?"
)

In [ ]:
# Test with a question that asks about investment recommendation, or hate or violence
test_agent_with_guardrail(
    "My SSN is 123-45-6789. Can you help me understand how to protect my retirement account?"
)

## Inspecting Conversation History

Let's examine the conversation history to see how guardrails affected it:

In [ ]:
# Print the conversation history
print(f"Conversation history: {json.dumps(agent.messages, indent=4)}")

## Integrating Bedrock guardrails to a chatbot 

Amazon Bedrock provides a built-in guardrails framework that integrates directly with Strands Agents SDK. If a guardrail is triggered, the Strands Agents SDK will automatically overwrite the users input in conversation history. This is done so that follow-up questions are not also blocked by the same questions. This can be configured with the guardrail_redact_input boolean, and the guardrail_redact_input_message string to change the overwrite message. Additionally, the same functionality is built for the model's output, but this is disabled by default. You can enable this with the guardrail_redact_output boolean, and change the overwrite message with the guardrail_redact_output_message string. Below is an example of how to leverage Bedrock guardrails in your code:

In [ ]:
bedrock_model = BedrockModel(
    model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    guardrail_id=guardrail_id,         # Your Bedrock guardrail ID
    guardrail_version=guardrail_version,             # Guardrail version
    guardrail_trace="enabled",
    guardrail_redact_output = True,        
    guardrail_redact_input = True  ,
    guardrail_redact_input_message = "Guardrail Intervened and Redacted"     
)

# Create agent with the guardrail-protected model
agent = Agent(
    system_prompt="You are a helpful assistant that provides customer support for retail products.",
    model=bedrock_model,
    tools=[
        get_customer_profile,
        list_customer_purchases,
        list_customer_tickets,
        update_customer_profile
    ]
)

In [ ]:
# Test a safe question for the customer support Agent
agent(f"what is the data from customer_id {sample_profile.customer_id}?")

In [ ]:
# Test with user input in which guardrail is intervened and input is redacted with custom message 
agent("How should I allocate my 401(k) investments?")

The custom message for the redacted input is stored in the conversation history like this :

In [ ]:
print(f"Conversation history: {json.dumps(agent.messages, indent=4)}")

## Clean up 

Please delete the guardrail as follows :

In [ ]:
bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)

## Congratulations!

In this notebook, we demonstrated how to:

1. Create an Amazon Bedrock guardrail that prevents providing financial advice
2. Test the guardrail directly using the Bedrock Runtime API
3. Integrate the guardrail with a Strands agent
4. Test the agent with various inputs to see the guardrail in action
5. Integrate bedrock guardrails in a chatbot with Strands agent.
5. Delete the Guardrail

Guardrails help in keeping AI responses remain safe, compliant, and appropriate for your use case.


---

# Exercise — The guardrail that was not watching

**Time: 10-15 minutes.**

Look again at the risky prompt you sent earlier in this lab:

> *"My SSN is 123-45-6789. Can you help me understand how to protect my retirement account?"*

The guardrail intervened, so it looked like a win. But it intervened on **Fiduciary Advice**, the
topic you denied. The social security number had nothing to do with it. Strip the financial question
and send the SSN on its own and the guardrail does not react at all: it has a topic policy, a content
policy and a word policy, and **no policy about personal data whatsoever**.

Your customer support agent will happily write that SSN into its conversation history.

**Write the missing policy.** Add a sensitive-information policy to the guardrail so that:

- a message containing a US social security number is **blocked**
- a message containing an email address is **anonymised** rather than blocked, because your support
  agent legitimately needs to discuss customers by email
- an ordinary support question still gets through untouched

The API you need is the `sensitiveInformationPolicyConfig` argument of `create_guardrail`. The
[PII entity types](https://docs.aws.amazon.com/bedrock/latest/userguide/guardrails-sensitive-filters.html)
are listed in the docs; you want `action` values of `BLOCK` and `ANONYMIZE`.

One practical note so you do not lose time on it: `BLOCK` rejects a user's **input**, while
`ANONYMIZE` rewrites the model's **output**. The self-check already tests each one in the right
direction.

In [ ]:
# ---- provided scaffold ----
import time
import uuid

import boto3

bedrock_client = boto3.client("bedrock")
bedrock_runtime = boto3.client("bedrock-runtime")

EX_SUFFIX = uuid.uuid4().hex[:8]        # keeps your guardrails separate from everyone else's
EX_GUARDRAILS = []                      # everything created here, so we can clean up

TOPIC_POLICY = {
    "topicsConfig": [
        {
            "name": "Fiduciary Advice",
            "definition": (
                "Providing personalized advice or recommendations on managing financial "
                "assets, investments, or trusts in a fiduciary capacity."
            ),
            "examples": [
                "What stocks should I invest in for my retirement?",
                "How should I allocate my 401(k) investments?",
            ],
            "type": "DENY",
        }
    ]
}


def make_guardrail(label, **extra_policies):
    """Create a guardrail with the fiduciary topic policy plus whatever you pass in."""
    response = bedrock_client.create_guardrail(
        name=f"exercise-{label}-{EX_SUFFIX}",
        description="Lab 06 exercise guardrail",
        topicPolicyConfig=TOPIC_POLICY,
        blockedInputMessaging="Blocked input.",
        blockedOutputsMessaging="Blocked output.",
        **extra_policies,
    )
    EX_GUARDRAILS.append(response["guardrailId"])
    return response["guardrailId"]


def check(guardrail_id, text, source="INPUT"):
    """Apply the guardrail to one piece of text and report what it did.

    source="INPUT" checks what the user typed; source="OUTPUT" checks what the
    model is about to say. The two are not interchangeable.
    """
    response = bedrock_runtime.apply_guardrail(
        guardrailIdentifier=guardrail_id,
        guardrailVersion="DRAFT",
        source=source,
        content=[{"text": {"text": text}}],
    )
    policies = set()
    for assessment in response.get("assessments", []):
        policies |= set(assessment.keys())
    outputs = response.get("outputs") or []
    return {
        "action": response.get("action"),
        "policies": sorted(p for p in policies if p.endswith("Policy")),
        "output": outputs[0].get("text", "") if outputs else "",
    }


def cleanup_exercise_guardrails():
    """Delete every guardrail this exercise created. Run this when you are done."""
    for guardrail_id in list(EX_GUARDRAILS):
        try:
            bedrock_client.delete_guardrail(guardrailIdentifier=guardrail_id)
            print("deleted", guardrail_id)
            EX_GUARDRAILS.remove(guardrail_id)
        except Exception as exc:
            print("could not delete", guardrail_id, exc)


SSN_ONLY = "My SSN is 123-45-6789. Please update my shipping address."
EMAIL_ONLY = "Please send the receipt to anna.dupont@example.com."
ORDINARY = "What was my latest purchase? My customer ID is CUST100."
print("scaffold ready")

In [ ]:
# ---- provided: the guardrail as this lab built it, tested on an SSN alone ----
topic_only = make_guardrail("topic-only")
time.sleep(8)          # give the new guardrail a moment to become usable

print("SSN on its own, against the lab's guardrail:")
print("  ", check(topic_only, SSN_ONLY))
print("\n-> action NONE means the number went straight through.")

In [ ]:
# ---------------------------------------------------------------------------
# YOUR WORK GOES HERE
#
# Build the sensitive-information policy: block US social security numbers,
# anonymise email addresses.
# ---------------------------------------------------------------------------

SENSITIVE_INFORMATION_POLICY = {
    # TODO: fill this in. Look up the shape of sensitiveInformationPolicyConfig.
}

if not SENSITIVE_INFORMATION_POLICY:
    raise NotImplementedError("Write the policy, then re-run this cell.")

my_guardrail = make_guardrail("with-pii", sensitiveInformationPolicyConfig=SENSITIVE_INFORMATION_POLICY)
time.sleep(8)

ssn_result = check(my_guardrail, SSN_ONLY)
email_result = check(my_guardrail, EMAIL_ONLY, source="OUTPUT")
ordinary_result = check(my_guardrail, ORDINARY)
for label, result in [("SSN", ssn_result), ("EMAIL", email_result), ("ORDINARY", ordinary_result)]:
    print(f"  {label:9} {result}")

In [ ]:
checks = {
    "a social security number is blocked":
        ssn_result["action"] == "GUARDRAIL_INTERVENED"
        and "sensitiveInformationPolicy" in ssn_result["policies"],
    "an email address is masked in what the model says, not blocked":
        email_result["action"] == "GUARDRAIL_INTERVENED"
        and "{EMAIL}" in email_result["output"],
    "an ordinary support question is untouched":
        ordinary_result["action"] == "NONE",
}
for label, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {label}")

print()
if all(checks.values()):
    print("Exercise complete. Now run the cleanup cell below.")
elif ordinary_result["action"] != "NONE":
    print("You are blocking normal traffic too. A guardrail nobody can talk through is not a guardrail.")
else:
    print("Not there yet - check the entity types and the action on each one.")

### Clean up

Guardrails are cheap but they are real resources. Delete the ones this exercise created.

In [ ]:
cleanup_exercise_guardrails()

## Stuck?

One possible solution is stored below, encoded so you do not read it by accident. Try the exercise
first: the point is the attempt, not the answer.

Run `show_solution()` to print it, or `load_solution()` to run it, then re-run the self-check.

In [ ]:
import base64

_SOLUTION = (
    "U0VOU0lUSVZFX0lORk9STUFUSU9OX1BPTElDWSA9IHsKICAgICJwaWlFbnRpdGllc0NvbmZpZyI6IFsKICAgICAgICB7"
    "InR5cGUiOiAiVVNfU09DSUFMX1NFQ1VSSVRZX05VTUJFUiIsICJhY3Rpb24iOiAiQkxPQ0sifSwKICAgICAgICB7InR5"
    "cGUiOiAiRU1BSUwiLCAiYWN0aW9uIjogIkFOT05ZTUlaRSJ9LAogICAgXQp9CgpteV9ndWFyZHJhaWwgPSBtYWtlX2d1"
    "YXJkcmFpbCgid2l0aC1waWkiLCBzZW5zaXRpdmVJbmZvcm1hdGlvblBvbGljeUNvbmZpZz1TRU5TSVRJVkVfSU5GT1JN"
    "QVRJT05fUE9MSUNZKQp0aW1lLnNsZWVwKDgpCgpzc25fcmVzdWx0ID0gY2hlY2sobXlfZ3VhcmRyYWlsLCBTU05fT05M"
    "WSkKZW1haWxfcmVzdWx0ID0gY2hlY2sobXlfZ3VhcmRyYWlsLCBFTUFJTF9PTkxZLCBzb3VyY2U9Ik9VVFBVVCIpCm9y"
    "ZGluYXJ5X3Jlc3VsdCA9IGNoZWNrKG15X2d1YXJkcmFpbCwgT1JESU5BUlkpCmZvciBsYWJlbCwgcmVzdWx0IGluIFso"
    "IlNTTiIsIHNzbl9yZXN1bHQpLCAoIkVNQUlMIiwgZW1haWxfcmVzdWx0KSwgKCJPUkRJTkFSWSIsIG9yZGluYXJ5X3Jl"
    "c3VsdCldOgogICAgcHJpbnQoZiIgIHtsYWJlbDo5fSB7cmVzdWx0fSIpCgojIFRocmVlIHRoaW5ncyB3b3J0aCB0YWtp"
    "bmcgYXdheToKIyAgICogQkxPQ0sgYW5kIEFOT05ZTUlaRSBhY3QgaW4gb3Bwb3NpdGUgZGlyZWN0aW9ucy4gQkxPQ0sg"
    "cmVqZWN0cyB0aGUgdXNlcidzIGlucHV0LgojICAgICBBTk9OWU1JWkUgcmV3cml0ZXMgdGhlIG1vZGVsJ3Mgb3V0cHV0"
    "LCByZXBsYWNpbmcgdGhlIHZhbHVlIHdpdGggYSBwbGFjZWhvbGRlcgojICAgICBzdWNoIGFzIHtFTUFJTH0uIEFza2lu"
    "ZyBmb3IgQU5PTllNSVpFIG9uIGFuIElOUFVUIGNoZWNrIHJldHVybnMgYWN0aW9uIE5PTkUsCiMgICAgIHdoaWNoIGxv"
    "b2tzIGV4YWN0bHkgbGlrZSBubyBwcm90ZWN0aW9uIGF0IGFsbC4KIyAgICogQ2hvb3NpbmcgYmV0d2VlbiB0aGVtIGlz"
    "IGEgcHJvZHVjdCBkZWNpc2lvbi4gQmxvY2tpbmcgZXZlcnkgbWVzc2FnZSBjb250YWluaW5nCiMgICAgIGFuIGVtYWls"
    "IGFkZHJlc3Mgd291bGQgYnJlYWsgYSBzdXBwb3J0IGFnZW50IHRoYXQgZXhpc3RzIHRvIGRpc2N1c3MgY3VzdG9tZXJz"
    "LgojICAgKiBUaGUgd2lkZXIgcG9pbnQ6IGEgZ3VhcmRyYWlsIHJlZnVzZXMgb25seSB3aGF0IHlvdSBkZXNjcmliZWQu"
    "IFRoZSB0b3BpYyBwb2xpY3kKIyAgICAgd2FzIHJlYWwgcHJvdGVjdGlvbiBhZ2FpbnN0IG9uZSB0aGluZywgYW5kIGl0"
    "IG1hZGUgdGhlIGdhcCBuZXh0IHRvIGl0IGludmlzaWJsZS4KIyAgICAgQ292ZXJhZ2UgaXMgYSBsaXN0IHlvdSBtYWlu"
    "dGFpbiwgbm90IGEgc3dpdGNoIHlvdSB0dXJuIG9uLg=="
)


def show_solution() -> None:
    """Print one possible solution to the exercise."""
    print(base64.b64decode(_SOLUTION).decode())


def load_solution() -> None:
    """Run the reference solution, then re-run the self-check cell to see it pass."""
    exec(base64.b64decode(_SOLUTION).decode(), globals())


print("Ready. Call show_solution() to read it, or load_solution() to run it.")